# Benchmarking AMAT NESTML models

Once we have completely validated the AMAT NESTML we can move onto benchmarking the model. Ensuring the NESTML ODE toolbox is processing this model is the most efficient way possible. Here we can build these large networks known as Brunel Balanced networks.  Brunel Balanced Network Model is a theoretical framework in computational neuroscience that describes how neural stability is maintained through the near-perfect cancellation of massive excitatory and inhibitory currents. This fluctuation-driven mechanism allows the model to replicate the asynchronous and irregular firing patterns observed in the cerebral cortex. By providing a unifying principle for E-I balance, the model helps explain network architecture, plasticity, and the emergence of coherent oscillations like gamma rhythms.

Source: https://www.bohrium.com/en/sciencepedia/feynman/keyword/brunel_balanced_network_model

NEST Tutorial: https://nest-simulator.readthedocs.io/en/v3.7/auto_examples/brunel_alpha_nest.html#id2


In [ ]:
import nest
import pynestml
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Mapping, Sequence
import numpy as np
import matplotlib.pyplot as plt
import importlib
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

# Build and validate the NESTML AMAT neuronal model 

In [ ]:
def build_nestml_model(
    source_path: Path,
    module_name: str,) -> tuple[str, str]:
    
    if not source_path.exists():
        raise FileNotFoundError(f"NESTML source not found: {source_path}")

    generated_module, generated_model = NESTCodeGeneratorUtils.generate_code_for( # nest call to compile nestml c++ script 
        str(source_path),
        module_name=module_name,
        logging_level="INFO",)

    try:
        nest.Install(generated_module)
        print(f"Successfully installed module: '{generated_module}'")
    except nest.kernel.NESTError:
        print(f"Module '{generated_module}' is already installed. Skipping installation step.")

    # Verify the MODEL name is loaded and recognized by the NEST engine
    if generated_model not in nest.Models():
        raise RuntimeError(
            f"Generation completed, but model '{generated_model}' is missing from nest.Models()."
        )
    
    # return module, name 
    return generated_module, generated_model

# defining global variables 
nest_model = "amat2_psc_exp"
nestml_source = Path("./neurons_nestml/amat_neuron.nestml")
nestml_module = "nestml_amat_module"
resolution_ms = 0.1

# calling build function
module_name, nestml_model = build_nestml_model(
    nestml_source,
    nestml_module,
)

print("Loaded module:", module_name)
print("Generated model:", nestml_model)
print("NESTML recordables:", nest.GetDefaults(nestml_model)["recordables"])

# NEST Parameter Config 

In [1]:
from dataclasses import dataclass
from typing import Literal, Mapping

# restricts so only modeltype can be the following 
ModelType = Literal["nest", "nestml"]

@dataclass(frozen=True) # pass AMATConfig into both nestml and nest configs, ensures params are kept the same 
class AMATConfig:
    
    pulse_times_ms: tuple[float, ...] = (150.0, 170.0, 300.0, 360.0)
    pulse_amplitude_pa: float = 1000.0
    pulse_width_ms: float = 1.0
    simulation_time_ms: float = 500.0

    tau_m_ms: float = 10.0
    gamma: float = 1.5
    B: float = 4.0

    E_L_mv: float = -70.0
    resting_threshold_mv: float = -65.0

    alpha_1_mv: float = 5.0
    alpha_2_mv: float = 0.0
    tau_1_ms: float = 10.0
    tau_2_ms: float = 200.0
    
    @property # turns it into a read-only attribute 
    def tau_v_ms(self) -> float:
        """Voltage-dependent threshold timescale."""
        if self.gamma <= 0:
            raise ValueError("gamma must be greater than zero because tau_v = gamma * tau_m.")
        return self.gamma * self.tau_m_ms

    @property # Fixed: removed the extra leading space
    def beta_per_ms(self) -> float: # Fixed: removed the extra leading space
        """Convert dimensionless B to the model parameter beta."""
        return self.B / self.tau_m_ms

@dataclass # writes boilerplate code for the classes e.g., init, rep, eq
class SimulationResult:
    model_name: str
    model_type: ModelType
    events: Mapping
    spikes: Mapping
    parameters: dict[str, float]
